# Step 0：全局参数配置

本 Notebook 是项目的统一配置入口。它只定义参数，不读取数据、不执行定价、不拟合模型，也不包含交易逻辑。

当前尚未确定的业务参数使用 `None`，后续取得明确设定后再集中补充。

## 01_data_processing 参数

原始数据、清洗范围、合约解析和标准化输出参数。


In [15]:
from pathlib import Path

PROJECT_PATH = Path.cwd().resolve()
UNDERLYING_DATA_PATH = Path('数据/标的_2026-06~07.csv')
FUTURE_DATA_PATH = UNDERLYING_DATA_PATH
OPTION_DATA_PATH = Path('数据/MO_2026-06~07.csv')
OUTPUT_PATH = PROJECT_PATH / 'outputs'
PROCESSED_DATA_PATH = OUTPUT_PATH / '01_processed_data'
START_DATE = '2026-06-01'
END_DATE = '2026-07-29'
TRADING_DAYS_PER_YEAR = 365
UNDERLYING_CODE = '000852.SH'
UNDERLYING_PRICE_FIELD = 'CLOSE'
FUTURE_PREFIX = 'IM'
FUTURE_PRICE_FIELD = 'CLOSE'
OPTION_PREFIX = 'MO'
OPTION_RAW_PRICE_FIELD = 'S_DQ_CLOSE'
FILTER_ZERO_VOLUME = False
EXPIRY_DATE_OVERRIDES = {'2606': '2026-06-22'}
SAVE_CSV = True


## 02_basic_functions 参数

02 使用 01 cell 已设置的期限与到期日参数，不另设默认值。


In [16]:
TRADING_DAYS_PER_YEAR = TRADING_DAYS_PER_YEAR  # 在 00 文件的 01 cell 中设置为 TRADING_DAYS_PER_YEAR=365
EXPIRY_DATE_OVERRIDES = EXPIRY_DATE_OVERRIDES  # 在 00 文件的 01 cell 中设置，当前含 2606 特殊到期日


## 03_repo_forward 参数

Repo、利率、主力期货和 Forward 输出参数。


In [ ]:
REPO_FORWARD_OUTPUT_PATH = OUTPUT_PATH / '03_repo_forward'  # OUTPUT_PATH 在 00 文件的 01 cell 中设置
REPO_RECALC_FREQ = 10
RISK_FREE_RATE = 0.015
MAIN_FUTURE_SELECTION = 'VOLUME'
SAVE_CSV = SAVE_CSV  # 在 00 文件的 01 cell 中设置为 SAVE_CSV=True


## 04_volatility_model 参数

IV 反解、曲面拟合、Greeks 及 04 输出参数。


In [18]:
VOL_MODEL = 'QUADRATIC'
MIN_OTM_OPTIONS_PER_EXPIRY = 6
MIN_OPTION_PRICE = 1e-8
MIN_IV = None
MAX_IV = 5.0
SVI_PARAMETER_BOUNDS = {'a': (None, None), 'b': (None, None), 'rho': (None, None), 'm': (None, None), 'sigma': (None, None)}
SKEW_METHOD = 'DERIVATIVE'
SKEW_OUTPUT_TAG = '25DELTA' if SKEW_METHOD == 'DELTA' else 'DERIVATIVE'
CURVATURE_OUTPUT_TAG = 'DERIVATIVE'
MODEL_SKEW_TAG = f'{VOL_MODEL}_SKEW-{SKEW_OUTPUT_TAG}_CURVATURE-{CURVATURE_OUTPUT_TAG}'
VOLATILITY_MODEL_OUTPUT_PATH = OUTPUT_PATH / f'04_volatility_model_{MODEL_SKEW_TAG}'
IV_CURVE_FIGURE_PATH = VOLATILITY_MODEL_OUTPUT_PATH / 'figures' / 'IV_curve'
VOL_SURFACE_DELTA_BASIS = 'FORWARD'
VOL_SURFACE_TARGET_DELTAS = {'90PUT': -0.10, '25CALL': 0.25, '75PUT': -0.25, '10CALL': 0.10}
ATM_LOG_MONEYNESS = 0.0
IV_SOLVER_LOWER_BOUND = 1e-6
IV_SOLVER_TOLERANCE = 1e-8
IV_SOLVER_MAX_ITERATIONS = 200
SAVE_CSV = SAVE_CSV  # 在 00 文件的 01 cell 中设置为 SAVE_CSV=True
SAVE_FIGURE = True
FIGURE_FORMAT = 'png'


## 05_skew_curvature 参数

05 使用 04 的模型口径，并单独设置期限结构输出路径。


In [19]:
SKEW_METHOD = SKEW_METHOD  # 在 00 文件的 04 cell 中设置为 SKEW_METHOD='DERIVATIVE'
MODEL_SKEW_TAG = MODEL_SKEW_TAG  # 在 00 文件的 04 cell 中由模型、Skew、Curvature 口径生成
SKEW_CURVATURE_OUTPUT_PATH = OUTPUT_PATH / f'05_skew_curvature_{MODEL_SKEW_TAG}'
SAVE_CSV = SAVE_CSV  # 在 00 文件的 01 cell 中设置为 SAVE_CSV=True
SAVE_FIGURE = SAVE_FIGURE  # 在 00 文件的 04 cell 中设置为 SAVE_FIGURE=True
FIGURE_FORMAT = FIGURE_FORMAT  # 在 00 文件的 04 cell 中设置为 FIGURE_FORMAT='png'


## 06_skew_strategy 参数

选券、仓位、取整、费用、保证金和归因参数。


In [ ]:
VOL_MODEL = VOL_MODEL  # 在 00 文件的 04 cell 中设置为 VOL_MODEL='QUADRATIC'
MODEL_SKEW_TAG = MODEL_SKEW_TAG  # 在 00 文件的 04 cell 中生成
SKEW_STRATEGY_OUTPUT_PATH = OUTPUT_PATH / f'06_skew_strategy_{MODEL_SKEW_TAG}'
STRATEGY_MAX_OPTIONS = 9
STRATEGY_TARGET_SKEW_ABS = 10_000.0
STRATEGY_MONTHLY_SKEW_DIRECTION = True
STRATEGY_HEDGE_THETA = False  # True: 优化时令 V_tau 接近 0；False: Theta 不进入优化目标
STRATEGY_POSITION_BOUND = 30
STRATEGY_MAX_DAILY_TRADE_PER_LEG = 30  # 单日每个期权合约的最大仓位变动张数
STRATEGY_MIN_OPTION_VOLUME = 100
STRATEGY_USE_PREMIUM_MARGIN_FILTER = True  # True: 要求候选期权权利金金额/卖方单张保证金不低于阈值
STRATEGY_MIN_PREMIUM_MARGIN_RATIO = 0.10  # 10%
STRATEGY_MAX_ABS_LOG_MONEYNESS = 0.25
STRATEGY_MAX_MODEL_IV = 1.50
STRATEGY_MIN_CANDIDATE_OPTIONS = 5
STRATEGY_MIN_DTE_DAYS = 10
STRATEGY_MAX_CANDIDATE_EXPIRIES = 3
STRATEGY_RIDGE = 0.30
STRATEGY_GROSS_POSITION_PENALTY = 0.002
STRATEGY_INTEGER_OPTION_POSITIONS = True  # True: 期权仓位取整；False: 保留连续仓位
STRATEGY_INTEGER_FUTURES_POSITIONS = True  # True: 期货仓位取整；False: 保留连续对冲仓位
STRATEGY_OPTION_MULTIPLIER = 100.0
STRATEGY_OPTION_FEE_PER_CONTRACT = 15.0
STRATEGY_FUTURES_MULTIPLIER = 200.0
STRATEGY_FUTURES_FEE_RATE = 0.000023  # IM 非当日平仓手续费率；t开仓、t+1平仓双边均使用
STRATEGY_FUTURES_MARGIN_RATE = 0.08
STRATEGY_OPTION_MARGIN_RATE = 0.12
STRATEGY_MINIMUM_MARGIN_RATE = 0.06
STRATEGY_TAYLOR_STEP = 1e-4  # 完整 T1/T2 数值梯度与 Hessian 的相对 bump 步长
# Shapley、完整 Taylor 与 Traditional Taylor 统一固定为七因子并显式包含 Smile Roll。


## 参数清单

执行下方单元格可检查所有公开参数、含义与当前默认值。`None` 表示该参数仍等待后续业务定义。

In [21]:
import pandas as pd

PARAMETER_DESCRIPTIONS = {
    'PROJECT_PATH': '项目目录',
    'UNDERLYING_DATA_PATH': '标的原始数据路径',
    'FUTURE_DATA_PATH': 'IM期货原始数据路径；当前与标的数据共用文件',
    'OPTION_DATA_PATH': 'MO期权原始数据路径',
    'OUTPUT_PATH': '通用输出目录',
    'PROCESSED_DATA_PATH': '标准化数据表输出目录',
    'REPO_FORWARD_OUTPUT_PATH': 'Repo与Forward结果输出目录',
    'MODEL_SKEW_TAG': '供04及后续相关模块复用的模型与Skew口径标签',
    'SKEW_OUTPUT_TAG': '输出目录使用的25DELTA或DERIVATIVE标签',
    'VOLATILITY_MODEL_OUTPUT_PATH': '带模型与Skew口径标签的04结果输出目录',
    'IV_CURVE_FIGURE_PATH': '每日每到期日IV曲线图片目录',
    'SKEW_CURVATURE_OUTPUT_PATH': '带模型与Skew口径标签的05结果输出目录',
    'START_DATE': '回测开始日期',
    'END_DATE': '回测结束日期',
    'TRADING_DAYS_PER_YEAR': '期限与Theta年化天数口径',
    'UNDERLYING_CODE': '标的指数代码',
    'UNDERLYING_PRICE_FIELD': '标的收盘价字段',
    'FUTURE_PREFIX': '股指期货代码前缀',
    'FUTURE_PRICE_FIELD': '期货收盘价字段',
    'MAIN_FUTURE_SELECTION': '主力期货选择依据',
    'OPTION_PREFIX': '股指期权代码前缀',
    'OPTION_RAW_PRICE_FIELD': '期权原始文件中的收盘价字段',
    'FILTER_ZERO_VOLUME': '是否删除成交量为零的期权记录',
    'REPO_RECALC_FREQ': 'Repo重新计算的交易日频率',
    'RISK_FREE_RATE': '固定无风险利率值接口',
    'EXPIRY_DATE_OVERRIDES': '节假日等特殊合约月份的到期日覆盖表',
    'VOL_MODEL': '波动率模型选择',
    'MIN_OTM_OPTIONS_PER_EXPIRY': '单日单到期最少OTM样本数',
    'MIN_OPTION_PRICE': '最小有效期权收盘价',
    'MIN_IV': '最小有效隐含波动率',
    'MAX_IV': '最大有效隐含波动率',
    'SVI_PARAMETER_BOUNDS': 'SVI五个参数的上下界',
    'SKEW_METHOD': 'Skew定义方法',
    'VOL_SURFACE_DELTA_BASIS': 'IV曲线定位和Skew使用的Delta口径',
    'VOL_SURFACE_TARGET_DELTAS': 'IV曲线中90P、75P、25C与10C的Forward Delta目标',
    'ATM_LOG_MONEYNESS': 'ATM Forward对应的log-moneyness位置',
    'IV_SOLVER_LOWER_BOUND': '隐含波动率二分搜索下界',
    'IV_SOLVER_TOLERANCE': '隐含波动率价格求解容差',
    'IV_SOLVER_MAX_ITERATIONS': '隐含波动率二分搜索最大迭代次数',
    'SAVE_CSV': '是否保存表格结果',
    'SAVE_FIGURE': '是否保存图片',
    'FIGURE_FORMAT': '图片文件格式',
    'SKEW_STRATEGY_OUTPUT_PATH': '06 策略结果目录',
    'STRATEGY_MAX_OPTIONS': '每日最多选择的期权合约数',
    'STRATEGY_TARGET_SKEW_ABS': '组合 Skew 风险目标的绝对值',
    'STRATEGY_MONTHLY_SKEW_DIRECTION': '是否按月份切换 Skew 目标方向',
    'STRATEGY_HEDGE_THETA': '是否在仓位优化中将组合 V_tau 设为接近 0',
    'STRATEGY_POSITION_BOUND': '单个期权仓位绝对上限',
    'STRATEGY_MAX_DAILY_TRADE_PER_LEG': '单日单个期权合约仓位调整的最大张数',
    'STRATEGY_MIN_OPTION_VOLUME': '策略候选期权的最低成交量',
    'STRATEGY_USE_PREMIUM_MARGIN_FILTER': '是否启用候选期权权利金金额/卖方单张保证金过滤',
    'STRATEGY_MIN_PREMIUM_MARGIN_RATIO': '候选期权最低权利金金额/卖方单张保证金比例',
    'STRATEGY_MAX_ABS_LOG_MONEYNESS': '候选期权最大绝对对数虚实值',
    'STRATEGY_MAX_MODEL_IV': '候选期权最大模型隐含波动率',
    'STRATEGY_MIN_CANDIDATE_OPTIONS': '单个到期日至少候选期权数',
    'STRATEGY_MIN_DTE_DAYS': '候选到期日最小剩余自然日数',
    'STRATEGY_MAX_CANDIDATE_EXPIRIES': '每日最多考察的近月到期日数',
    'STRATEGY_RIDGE': '风险目标最小二乘的岭惩罚系数',
    'STRATEGY_GROSS_POSITION_PENALTY': '总持仓规模惩罚系数',
    'STRATEGY_OPTION_MULTIPLIER': '策略期权合约乘数',
    'STRATEGY_OPTION_FEE_PER_CONTRACT': '期权每张单边交易费',
    'STRATEGY_FUTURES_MULTIPLIER': 'IM 期货合约乘数',
    'STRATEGY_FUTURES_FEE_RATE': 'IM非当日平仓手续费率；按成交金额计算',
    'STRATEGY_FUTURES_MARGIN_RATE': 'IM 期货保证金率',
    'STRATEGY_OPTION_MARGIN_RATE': '卖出期权保证金中的标的保证金率',
    'STRATEGY_MINIMUM_MARGIN_RATE': '卖出期权最低保证金率',
    'STRATEGY_INTEGER_OPTION_POSITIONS': '是否将06策略期权仓位取整',
    'STRATEGY_INTEGER_FUTURES_POSITIONS': '是否将06策略期货对冲仓位取整',
    'STRATEGY_TAYLOR_STEP': 'Taylor 数值 Hessian 的相对差分步长',
}

parameter_table = pd.DataFrame([
    {
        '参数': name,
        '含义': description,
        '默认值': repr(globals()[name]),
        '状态': '待配置' if globals()[name] is None else '已设置',
    }
    for name, description in PARAMETER_DESCRIPTIONS.items()
])

print(f'配置参数数量: {len(parameter_table)}')
display(parameter_table)

配置参数数量: 69


,参数,含义,默认值,状态
0,PROJECT_PATH,项目目录,PosixPath('/Users/mac/Desktop/实习/skew_strategy...,已设置
1,UNDERLYING_DATA_PATH,标的原始数据路径,PosixPath('数据/标的_2026-06~07.csv'),已设置
2,FUTURE_DATA_PATH,IM期货原始数据路径；当前与标的数据共用文件,PosixPath('数据/标的_2026-06~07.csv'),已设置
3,OPTION_DATA_PATH,MO期权原始数据路径,PosixPath('数据/MO_2026-06~07.csv'),已设置
4,OUTPUT_PATH,通用输出目录,PosixPath('/Users/mac/Desktop/实习/skew_strategy...,已设置
...,...,...,...,...
64,STRATEGY_OPTION_MARGIN_RATE,卖出期权保证金中的标的保证金率,0.12,已设置
65,STRATEGY_MINIMUM_MARGIN_RATE,卖出期权最低保证金率,0.06,已设置
66,STRATEGY_INTEGER_OPTION_POSITIONS,是否将06策略期权仓位取整,True,已设置
67,STRATEGY_INTEGER_FUTURES_POSITIONS,是否将06策略期货对冲仓位取整,True,已设置
